# Day 14 — Subqueries + EXISTS + IN
**Estimated time:** 60–75 min
**Dataset:** `sales_orders.csv`

## Learning Objectives
- Write scalar, correlated, and derived-table subqueries
- Understand the semantic and performance differences between IN and EXISTS
- Use NOT EXISTS as a reliable anti-join pattern (safer than NOT IN with NULLs)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = "../../data/raw"

# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- Scalar subquery: revenue share per sales org --
result = q(
    """
    SELECT
        VKORG,
        SUM(NETWR) AS org_revenue,
        (SELECT SUM(NETWR) FROM SALES) AS total_revenue,
        ROUND(SUM(NETWR) * 100.0 / (SELECT SUM(NETWR) FROM SALES), 2) AS pct_of_total
    FROM SALES
    GROUP BY VKORG
    ORDER BY org_revenue DESC
    """
)
display(result)

In [ ]:
# -- Subquery in WHERE (IN list) --
top_vkorg = q("SELECT VKORG FROM SALES GROUP BY VKORG ORDER BY SUM(NETWR) DESC LIMIT 1").iloc[0,0]
print(f"Top VKORG by revenue: {top_vkorg}")

result = q(f"""
    SELECT KUNNR, COUNT(*) AS orders, SUM(NETWR) AS revenue
    FROM SALES
    WHERE KUNNR IN (
        SELECT DISTINCT KUNNR FROM SALES WHERE VKORG = '{top_vkorg}'
    )
    GROUP BY KUNNR
    ORDER BY revenue DESC
    LIMIT 10
""")
display(result)

In [ ]:
# -- Correlated subquery: each order as pct of customer total --
result = q(
    """
    SELECT
        VBELN,
        KUNNR,
        NETWR,
        ROUND(
            NETWR * 100.0 / (
                SELECT SUM(NETWR) FROM SALES inner_s
                WHERE inner_s.KUNNR = outer_s.KUNNR
            )
        , 2) AS pct_of_customer_total
    FROM SALES outer_s
    WHERE NETWR > 0
    ORDER BY pct_of_customer_total DESC
    LIMIT 15
    """
)
display(result)

In [ ]:
# -- Derived table (subquery in FROM): high-cancel-rate customers --
result = q(
    """
    SELECT *
    FROM (
        SELECT
            KUNNR,
            COUNT(DISTINCT VBELN)  AS total_orders,
            SUM(NETWR)             AS total_revenue,
            SUM(CASE WHEN STATUS = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_count,
            ROUND(
                SUM(CASE WHEN STATUS = 'CANCELLED' THEN 1.0 ELSE 0 END)
                / COUNT(DISTINCT VBELN) * 100
            , 2) AS cancel_rate_pct
        FROM SALES
        GROUP BY KUNNR
    ) customer_summary
    WHERE total_orders >= 3
      AND cancel_rate_pct > 20
    ORDER BY cancel_rate_pct DESC
    """
)
print(f"High-risk customers (cancel_rate > 20%, min 3 orders): {len(result)}")
display(result)

In [ ]:
# -- IN vs EXISTS: semantic and performance comparison --
# IN: evaluates subquery to a list, then checks membership
# EXISTS: short-circuits as soon as the first match is found

result_in = q(
    """
    SELECT DISTINCT KUNNR FROM SALES
    WHERE KUNNR IN (
        SELECT KUNNR FROM SALES WHERE ERDAT >= '2024-01-01'
    )
    """
)
result_exists = q(
    """
    SELECT DISTINCT s1.KUNNR FROM SALES s1
    WHERE EXISTS (
        SELECT 1 FROM SALES s2
        WHERE s2.KUNNR = s1.KUNNR AND s2.ERDAT >= '2024-01-01'
    )
    """
)
print(f"IN result: {len(result_in)} customers | EXISTS result: {len(result_exists)} customers")
assert len(result_in) == len(result_exists)
print("Results are identical.")

In [ ]:
# -- NOT IN vs NOT EXISTS: the NULL trap --
# NOT IN: if the subquery returns ANY NULL value, the whole NOT IN is FALSE (returns 0 rows!)
# NOT EXISTS: correctly handles NULLs in all cases
#
# Safe anti-join with NOT EXISTS:
result = q(
    """
    SELECT DISTINCT KUNNR FROM SALES s_outer
    WHERE ERDAT >= '2024-01-01'
      AND NOT EXISTS (
          SELECT 1 FROM SALES s_inner
          WHERE s_inner.KUNNR = s_outer.KUNNR
            AND s_inner.ERDAT >= '2025-01-01'
      )
    ORDER BY KUNNR
    """
)
print(f"Customers active in 2024 but NOT in 2025 (churned): {len(result)}")
display(result.head(10))

---
## Daily Prompt

**Find customers who placed orders in 2024 but have NOT placed any orders in 2025. Use BOTH approaches and compare the row counts.**

```sql
-- Approach 1: NOT IN (watch for NULL trap)
SELECT DISTINCT KUNNR
FROM SALES
WHERE ERDAT >= '2024-01-01' AND ERDAT < '2025-01-01'
  AND KUNNR NOT IN (
      -- YOUR SOLUTION: SELECT KUNNR from 2025 orders (add WHERE KUNNR IS NOT NULL!)
  )
ORDER BY KUNNR
```

```sql
-- Approach 2: NOT EXISTS (preferred — NULL-safe)
SELECT DISTINCT s1.KUNNR
FROM SALES s1
WHERE s1.ERDAT >= '2024-01-01' AND s1.ERDAT < '2025-01-01'
  AND NOT EXISTS (
      -- YOUR SOLUTION
  )
ORDER BY KUNNR
```

> **The NULL trap:** Add `WHERE KUNNR IS NOT NULL` inside NOT IN to prevent silent zero-row results.
> Commercial context: these are churned customers — prioritize for win-back campaigns.